In [ ]:
# pip install pandas scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# --- Step 1: Download dataset directly (UCI, no auth needed) ---
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')


In [ ]:
# --- Step 2: Turn quality score (3-8) into 3 classes ---
def quality_to_class(q):
    if q <= 4:
        return 'low'
    elif q <= 6:
        return 'medium'
    else:
        return 'high'

df['quality_class'] = df['quality'].apply(quality_to_class)

X = df.drop(columns=['quality', 'quality_class'])
y = df['quality_class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# --- Step 4: Train Multinomial Logistic Regression ---
model = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                            max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print("--- Multinomial Logistic Regression ---")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Wine Quality")
plt.show()

ValueError: The 'liblinear' solver does not support multiclass classification (n_classes >= 3). Either use another solver or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.

In [ ]:
# --- Step 5: Compare with One-vs-Rest ---
model_ovr = LogisticRegression(multi_class='ovr', solver='liblinear',
                                max_iter=1000, class_weight='balanced')
model_ovr.fit(X_train_scaled, y_train)
y_pred_ovr = model_ovr.predict(X_test_scaled)

print("--- One-vs-Rest ---")
print(classification_report(y_test, y_pred_ovr))

In [ ]:
# --- Step 6: Make a Manual Prediction on New Data ---
# Creating a dummy wine sample with 11 chemical features (must match X columns)
# Features: fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, 
# free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, alcohol
new_wine = np.array([[7.4, 0.70, 0.00, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]])

# 1. Scale the new data using the ALREADY FITTED scaler
new_wine_scaled = scaler.transform(new_wine)

# 2. Predict the class using the Multinomial model
manual_prediction = model.predict(new_wine_scaled)

# 3. View the underlying probabilities for each class
probabilities = model.predict_proba(new_wine_scaled)

print(f"Predicted Class: {manual_prediction[0]}")
print(f"Class Probabilities {model.classes_}: {probabilities[0]}")

In [ ]:
'''

Here's a line-by-line breakdown:

python
model = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                            max_iter=1000, class_weight='balanced')
Creates a Logistic Regression model object (not trained yet).
multi_class='multinomial' → since you have 3 classes (low/medium/high), this tells sklearn to train one combined model that directly predicts probabilities across all classes at once (using the softmax function), instead of building 3 separate binary "one vs rest" classifiers. This usually gives better-calibrated probabilities for multiclass problems.
solver='lbfgs' → the optimization algorithm used to find the best coefficients. lbfgs is required (or one of a few options) when using multinomial — it's a solid general-purpose default.
max_iter=1000 → maximum number of iterations the solver is allowed to run while trying to converge on a solution. Default is 100, which is often too low and throws a "failed to converge" warning, so it's bumped up.
class_weight='balanced' → since your classes are likely imbalanced (medium quality wines are probably way more common than low/high), this automatically re-weights the loss function so the model doesn't just lazily predict "medium" every time. It penalizes mistakes on minority classes more heavily.
python
model.fit(X_train_scaled, y_train)
This is where actual training happens. The model looks at your scaled training features (X_train_scaled) and the true labels (y_train), and learns the coefficients that best separate the 3 classes.
python
y_pred = model.predict(X_test_scaled)
Uses the now-trained model to predict class labels (low/medium/high) for the test set it has never seen before. Output is an array of predicted class labels, same length as y_test.
python
print(classification_report(y_test, y_pred))
Compares true labels (y_test) vs predicted labels (y_pred) and prints a table showing precision, recall, f1-score, and support for each class individually, plus overall accuracy and macro/weighted averages. This tells you not just "how accurate" but where the model struggles (e.g., maybe it's great at "medium" but terrible at "low" since there's less data).
python
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
Builds a confusion matrix: a grid showing actual class (rows) vs predicted class (columns), with counts of how many predictions fell into each combination.
labels=model.classes_ ensures the rows/columns are ordered consistently according to how the model internally orders the classes (alphabetical: high, low, medium), so nothing gets mismatched.
python
sns.heatmap(cm, annot=True, fmt='d', xticklabels=model.classes_, yticklabels=model.classes_)
Draws the confusion matrix as a color-coded heatmap using Seaborn, instead of just a plain grid of numbers — much easier to visually spot where the model confuses classes.
annot=True → shows the actual number in each cell.
fmt='d' → formats those numbers as integers (not scientific notation/decimals).
xticklabels / yticklabels → labels the axes with the actual class names (high/low/medium) instead of just 0/1/2.
python
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Wine Quality")
plt.show()
Standard matplotlib labeling: names the x-axis, y-axis, and chart title, then renders the plot on screen.
python
model_ovr = LogisticRegression(multi_class='ovr', solver='liblinear',
                                max_iter=1000, class_weight='balanced')
Creates a second model, this time using the One-vs-Rest (OvR) strategy instead of multinomial. This trains 3 separate binary classifiers under the hood (e.g., "low vs not-low", "medium vs not-medium", "high vs not-high") and picks whichever one is most confident for each prediction.
solver='liblinear' → a different optimizer, commonly paired with OvR (and required for some penalty types); works well on smaller datasets like this one.
python
model_ovr.fit(X_train_scaled, y_train)
Trains this second OvR model the same way as before.
python
y_pred_ovr = model_ovr.predict(X_test_scaled)
Generates predictions from the OvR model on the same test set, so you can directly compare it against the multinomial model's performance.
python
print("--- One-vs-Rest ---")
print(classification_report(y_test, y_pred_ovr))
Prints a header so the output is readable, then prints the same precision/recall/f1 report — but this time for the OvR model — so you can compare the two strategies side by side and see which one performs better on your specific dataset.'''